# Esperimento 03b — separabilità o supervisione condivisa?

La coordinata 8 locale era più accurata, ma il sistema globale peggiorava. Questo esperimento distingue due spiegazioni usando multi-task learning standard: il candidato mantiene l'output locale e aggiunge una predizione globale ausiliaria della coordinata 8, usata soltanto nella loss. Un controllo aggiunge quasi la stessa capacità alla testa globale senza auxiliary target.

In [ ]:
from pathlib import Path
import subprocess,sys,tempfile
def project_root():
    candidates=[Path.cwd(),Path.cwd().parent]; work=Path('/kaggle/working')
    if work.exists(): candidates += [p.parent for p in work.glob('*/pyproject.toml')]
    for candidate in candidates:
        marker=candidate/'pyproject.toml'
        if marker.exists() and 'hay-single-compartment' in marker.read_text(): return candidate
    destination=Path(tempfile.mkdtemp(prefix='hay_composite_03b_',dir='/kaggle/working'))
    subprocess.check_call(['git','clone','--depth','1','https://github.com/Zagred47/LearningSingleCompartiment.git',str(destination)]); return destination
ROOT=project_root(); SRC=ROOT/'src'; assert (SRC/'hay_single_compartment').is_dir()
sys.path.insert(0,str(SRC)); print('Project:',ROOT)

In [ ]:
import h5py,json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from hay_single_compartment import INPUT_NAMES,STATE_NAMES,SimulationConfig,generate_dataset,validate_dataset
from hay_single_compartment.dataset import Normalization
from hay_single_compartment.models import build_model
from hay_single_compartment.training import rollout_batch,train_model
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR=Path('/kaggle/working/hay_composite_experiment_03b') if Path('/kaggle').exists() else ROOT/'artifacts'/'composite_03b'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True); DATASET=OUTPUT_DIR/'single_compartment_composite_v1.h5'
print('Device:',DEVICE,'| output:',OUTPUT_DIR)

In [ ]:
config=SimulationConfig(duration_ms=500.,warmup_ms=150.,seed=27182,train_trajectories=24,validation_trajectories=4,test_trajectories=6)
dataset_report=generate_dataset(DATASET,config,progress=True) if not DATASET.exists() else validate_dataset(DATASET)
dataset_report

## Quattro condizioni, un solo dataset

`accepted_composite` è il vincitore prima della separazione. `local_8` riproduce il fallimento. `local_8_capacity` allarga la testa globale residua. `local_8_auxiliary` usa lo stesso output locale ma ripristina sul backbone il target 8 con peso `1/17`, uguale al contributo originario di una coordinata.

In [ ]:
COMMON=dict(hidden_dim=128,layers=3,width_multiplier=2,receptor_hidden_dim=32,receptor_layers=1,hcn_hidden_dim=32,hcn_layers=1,auxiliary_hidden_dim=32)
EXPERIMENTS={
 'accepted_composite':dict(architecture='conv_lstm_receptor_gru',global_head_dim=None),
 'local_8':dict(architecture='conv_lstm_receptor_hcn_gru',global_head_dim=None),
 'local_8_capacity':dict(architecture='conv_lstm_receptor_hcn_gru',global_head_dim=287),
 'local_8_auxiliary':dict(architecture='conv_lstm_receptor_hcn_aux',global_head_dim=None),
}
for settings in EXPERIMENTS.values():
    kwargs={**COMMON,**{k:v for k,v in settings.items() if k!='architecture'}}
    probe=build_model(settings['architecture'],len(STATE_NAMES)+len(INPUT_NAMES),len(STATE_NAMES),**kwargs)
    settings['parameters']=sum(p.numel() for p in probe.parameters())
parameter_table=pd.DataFrame(EXPERIMENTS).T[['architecture','parameters']]; parameter_table

In [ ]:
reports=[]
for name,settings in EXPERIMENTS.items():
    print('\n'+'='*100); print(f'Training {name}: {settings["parameters"]:,} parameters')
    reports.append(train_model(
        DATASET,OUTPUT_DIR/'models',settings['architecture'],run_name=name,train_fraction=1.,
        epochs=40,sequence_length=128,stride=32,batch_size=32,learning_rate=6e-4,dropout=.1,
        patience=8,minimum_epochs=18,device=DEVICE,seed=27182,use_amp=True,verbose=True,
        global_head_dim=settings.get('global_head_dim'),auxiliary_weight=1/17,**COMMON,
    ))
print('Diagnostic training completed.')

In [ ]:
comparison=pd.DataFrame([{
 'model':r['run_name'],'parameters':r['parameters'],'epochs':r['epochs_trained'],'validation_loss':r['best_validation_loss'],
 'test_voltage_rmse_mV':r['test']['voltage_rmse_mv'],'test_normalized_rmse':r['test']['mean_normalized_rmse'],
 'coordinate_8_rmse':r['test']['per_state_rmse'][STATE_NAMES[8]],'auxiliary_weight':r['auxiliary_weight'],
} for r in reports]).sort_values('validation_loss')
comparison.to_csv(OUTPUT_DIR/'comparison.csv',index=False); comparison

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
for report in reports:
    history=pd.DataFrame(report['history']); axes[0].plot(history.epoch,history.validation_loss,label=report['run_name']); axes[1].plot(history.epoch,history.train_loss,label=report['run_name'])
axes[0].set(title='Validation main loss',xlabel='epoch',ylabel='loss',yscale='log'); axes[1].set(title='Training objective',xlabel='epoch',ylabel='loss',yscale='log')
for axis in axes: axis.grid(alpha=.25); axis.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'training_curves.png',dpi=160)

In [ ]:
entity_errors=pd.DataFrame([{'model':r['run_name'],'entity':e,'normalized_rmse':v} for r in reports for e,v in r['test']['per_group_normalized_rmse'].items()])
entity_errors.to_csv(OUTPUT_DIR/'entity_errors.csv',index=False)
entity_errors.pivot(index='entity',columns='model',values='normalized_rmse').sort_values('local_8_auxiliary',ascending=False)

In [ ]:
with h5py.File(DATASET,'r') as h5: truth=h5['test/states'][...]; future_inputs=h5['test/inputs'][...]
rollout_rows=[]; predictions={}
for report in reports:
    checkpoint=torch.load(report['checkpoint'],map_location=DEVICE,weights_only=False)
    model=build_model(checkpoint['architecture'],len(STATE_NAMES)+len(INPUT_NAMES),len(STATE_NAMES),**checkpoint['model_kwargs']).to(DEVICE)
    model.load_state_dict(checkpoint['model_state']); norm=Normalization.from_dict(checkpoint['normalization'])
    print(f'\nRollout {report["run_name"]}...'); prediction=rollout_batch(model,truth[:,0],future_inputs,norm,DEVICE,progress=True); predictions[report['run_name']]=prediction
    for horizon in (50,100,200,500):
        end=int(horizon/config.dt_ms)+1; error=prediction[:,:end]-truth[:,:end]
        ps=((prediction[:,1:end,0]>=config.membrane.spike_threshold_mv)&(prediction[:,:end-1,0]<config.membrane.spike_threshold_mv)).sum(); ts=((truth[:,1:end,0]>=config.membrane.spike_threshold_mv)&(truth[:,:end-1,0]<config.membrane.spike_threshold_mv)).sum()
        rollout_rows.append({'model':report['run_name'],'horizon_ms':horizon,'voltage_rmse_mV':float(np.sqrt(np.mean(error[...,0]**2))),'mean_normalized_rmse':float(np.sqrt(np.mean((error/norm.state_std)**2,axis=(0,1))).mean()),'teacher_spikes':int(ts),'predicted_spikes':int(ps)})
rollout_table=pd.DataFrame(rollout_rows); rollout_table.to_csv(OUTPUT_DIR/'rollout_comparison.csv',index=False); rollout_table

In [ ]:
time_ms=np.arange(truth.shape[1])*config.dt_ms; fig,axes=plt.subplots(2,1,figsize=(15,7),sharex=True)
for axis,index,label in zip(axes,(0,8),('state 0','state 8')):
    axis.plot(time_ms,truth[0,:,index],color='black',label='teacher',lw=1.2)
    for name,prediction in predictions.items(): axis.plot(time_ms,prediction[0,:,index],label=name,alpha=.8)
    axis.set_ylabel(label); axis.grid(alpha=.2); axis.legend()
axes[-1].set_xlabel('time (ms)'); fig.suptitle('Auxiliary-supervision diagnostic')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'rollout_example.png',dpi=160)

## Decisione

Se l'auxiliary head recupera le coordinate globali e il rollout meglio del capacity control, la separazione locale era compatibile ma la supervisione condivisa era necessaria. Se non li recupera, la fattorizzazione della coordinata 8 viene rifiutata definitivamente nella forma testata.

In [ ]:
from shutil import copytree,make_archive,rmtree
import base64,os
from IPython.display import Javascript,display
parameter_table.to_csv(OUTPUT_DIR/'parameter_table.csv'); (OUTPUT_DIR/'experiment_definition.json').write_text(json.dumps({'experiments':EXPERIMENTS,'common':COMMON,'auxiliary_weight':1/17,'seed':config.seed},indent=2),encoding='utf-8')
include_checkpoints=os.environ.get('HAY_DOWNLOAD_CHECKPOINTS','0')=='1'; archive_source=OUTPUT_DIR; staging=Path('/kaggle/working/hay_composite_03b_download')
if not include_checkpoints:
    if staging.exists(): rmtree(staging)
    copytree(OUTPUT_DIR,staging,ignore=lambda path,names:{n for n in names if n.endswith('.pt')}); archive_source=staging
zip_path=Path(make_archive('/kaggle/working/hay_composite_experiment_03b_complete','zip',root_dir=archive_source.parent,base_dir=archive_source.name)); encoded=base64.b64encode(zip_path.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{encoded}'),x=new Uint8Array(b.length);for(let i=0;i<b.length;i++)x[i]=b.charCodeAt(i);const o=URL.createObjectURL(new Blob([x],{{type:'application/zip'}})),a=document.createElement('a');a.href=o;a.download='{zip_path.name}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(o),60000);"""))
print('Download avviato:',zip_path,f'({zip_path.stat().st_size/2**20:.1f} MiB)','| checkpoint inclusi:',include_checkpoints)